# BQ2 · XGBoost: which laundering schemes need the network to be seen?

**Business question 2:** where does money laundering concentrate, and how does each laundering scheme move money?

This notebook trains one XGBoost model on IBM AML HI-Small twice:

- **Experiment A**: per-transaction and one-hop account features (the baseline feature set).
- **Experiment B**: A plus five network features that follow money across more than one hop.

The comparison that answers BQ2 is the **detection rate per laundering scheme** in the top 0.5% of scores. It shows which schemes only become visible once the model looks at the network.

Spec: `notebooks/AGENT_HANDOVER.md`. Outputs go to `out/model/` (section 11).

**Run:** select the repo's `.venv` kernel, then use *Restart & Run All*. Set `AML_MODE=aws` to read the Athena `curated/transactions` Parquet instead of the local files.

## 1. Setup

In [ ]:
import json
import math
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import shap
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_recall_curve

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
T0 = time.time()

# ---- config: AML_MODE is the only switch between the laptop and AWS ----
AML_MODE = os.environ.get("AML_MODE", "local")                       # "local" | "aws"
DATA_DIR = Path(os.environ.get("AML_DATA_DIR", "~/aml_data")).expanduser()
RAW_TRANS = DATA_DIR / "HI-Small_Trans.csv"                          # local: the Kaggle files
RAW_PATTERNS = DATA_DIR / "HI-Small_Patterns.txt"
DATA_PATH = DATA_DIR / "aml_out" / "hi_small_clean.csv"              # local fallback (see section 2)

REPO = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
# aws: aws s3 sync s3://<bucket>/curated/ ./curated/ --profile aml-bq2
CURATED_DIR = Path(os.environ.get("AML_CURATED_DIR", REPO / "curated" / "transactions"))
OUT_DIR = REPO / "out" / "model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
ALERT_SHARE = 0.005
VALID_START = pd.Timestamp("2022-09-07")   # fit 1-6 Sep, valid 7 Sep (early stopping only)
TEST_START = pd.Timestamp("2022-09-08")    # test 8-10 Sep, touched only for the final evaluation
TAIL_START = pd.Timestamp("2022-09-11")    # 11-18 Sep tail is 57-73% laundering: always dropped

# Measured facts from the handover (section 2); the load cell asserts them
EXPECTED = {"rows_after_dedup": 5_078_336, "laundering": 5_177, "accounts": 515_080,
            "train_rows": 3_731_669, "train_laundering": 3_027, "test_laundering": 1_495}
print(f"mode={AML_MODE}  xgboost {xgb.__version__}  shap {shap.__version__}  out={OUT_DIR}")

## 2. Load and check

**Inputs**

- **`local`** reads the raw Kaggle files (`HI-Small_Trans.csv` and `HI-Small_Patterns.txt`), with bank and account ids as strings. If they are missing, it falls back to `hi_small_clean.csv`. The raw files come first because `scripts/profile_aml.py` reads bank ids as integers. The clean file therefore stores `10` where S3 and the dashboard have `010`. No two banks collide (30,470 either way), but the bank keys would not join to the S3 data.
- **`aws`** reads the Athena `curated_transactions` Parquet, which carries `amount_paid_usd` and `pattern`.

**Scheme labels** are joined exactly as in `sql/02_curated_transactions.sql`: on `(ts, from_account, to_account, amount_paid)`, taking one label per transfer. Laundering rows without a scheme become `UNSTRUCTURED`; all other rows become `NONE`.

**USD conversion.** The IBM simulator converts currencies at its own fixed rates (for example 105.4 Yen per USD), not at 2022 market rates. Local mode therefore recovers those rates from the cross-currency rows that have USD on one side. This is the same method as `scripts/build_fx_from_data.py`, which built the `fx_rates` table behind `amount_paid_usd`, so both modes produce the same `amount_usd`.

Only the base columns are used. The precomputed account columns in `hi_small_clean.csv` cover all 10 days and would leak test behaviour into training, so every feature is recomputed below.

In [ ]:
USD = "US Dollar"
RAW_NAMES = ["ts", "from_bank", "from_account", "to_bank", "to_account", "amount_received",
             "receiving_currency", "amount_paid", "payment_currency", "payment_format", "is_laundering"]
ID_COLS = ["from_bank", "from_account", "to_bank", "to_account"]       # text: bank ids have leading zeros
LABEL_KEY = ["ts", "from_account", "to_account", "amount_paid"]


def read_raw_trans(path):
    opts = pacsv.ConvertOptions(column_types={**{c: pa.string() for c in ID_COLS}, "ts": pa.timestamp("s")},
                                timestamp_parsers=["%Y/%m/%d %H:%M"])
    tbl = pacsv.read_csv(path, read_options=pacsv.ReadOptions(column_names=RAW_NAMES, skip_rows=1),
                         convert_options=opts)
    df = tbl.to_pandas(strings_to_categorical=True)
    df["ts"] = df["ts"].astype("datetime64[ns]")
    return df


def read_patterns(path):
    begin = re.compile(r"^BEGIN LAUNDERING ATTEMPT\s*-\s*(.+?)\s*$", re.I)
    rows, attempt, pattern = [], -1, None
    with open(path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.strip()
            m = begin.match(line)
            if m:
                attempt += 1
                pattern = m.group(1).split(":")[0].strip().upper()
                continue
            if line.upper().startswith("END LAUNDERING ATTEMPT"):
                pattern = None
                continue
            parts = line.split(",")
            if pattern is not None and len(parts) == 11:
                rows.append((attempt, pattern, parts[0], parts[2], parts[4], float(parts[7])))
    pat = pd.DataFrame(rows, columns=["attempt_id", "pattern"] + LABEL_KEY)
    pat["ts"] = pd.to_datetime(pat["ts"], format="%Y/%m/%d %H:%M")
    # one label per transfer, as in the curated SQL: min(pattern), min(attempt_id)
    return pat.groupby(LABEL_KEY, as_index=False).agg(pattern=("pattern", "min"), attempt_id=("attempt_id", "min"))


def attach_patterns(df, pat):
    """Label only laundering rows (patterns never match clean rows), the rest are NONE."""
    lab = df.loc[df["is_laundering"] == 1, LABEL_KEY].astype({"from_account": str, "to_account": str})
    lab = lab.reset_index().merge(pat, on=LABEL_KEY, how="left").set_index("index")
    df["pattern"] = "NONE"
    df.loc[lab.index, "pattern"] = lab["pattern"].fillna("UNSTRUCTURED").values
    df["attempt_id"] = np.nan
    df.loc[lab.index, "attempt_id"] = lab["attempt_id"].values
    return df


def implied_usd_rates(df):
    """Units of each currency per USD, from rows with USD on exactly one side (median)."""
    pay, rec = df["payment_currency"].astype(str), df["receiving_currency"].astype(str)
    a, b = df[(rec == USD) & (pay != USD)], df[(pay == USD) & (rec != USD)]
    obs = pd.concat([pd.DataFrame({"currency": pay[a.index], "rate": a["amount_paid"] / a["amount_received"]}),
                     pd.DataFrame({"currency": rec[b.index], "rate": b["amount_received"] / b["amount_paid"]})])
    rates = obs[np.isfinite(obs["rate"]) & (obs["rate"] > 0)].groupby("currency")["rate"].median()
    rates[USD] = 1.0
    return rates.sort_index()


def load_local():
    if RAW_TRANS.exists() and RAW_PATTERNS.exists():
        df = read_raw_trans(RAW_TRANS)
        n_raw = len(df)
        df = df.drop_duplicates().reset_index(drop=True)
        df = attach_patterns(df, read_patterns(RAW_PATTERNS))
        src = f"{RAW_TRANS.name} + {RAW_PATTERNS.name}"
    else:
        cols = RAW_NAMES + ["pattern", "attempt_id"]
        df = pd.read_csv(DATA_PATH, usecols=cols, dtype={c: str for c in ID_COLS}, parse_dates=["ts"])
        n_raw = None
        df = df.drop_duplicates(subset=RAW_NAMES).reset_index(drop=True)
        none = df["pattern"].isna() | (df["pattern"] == "NONE")
        df.loc[none, "pattern"] = np.where(df.loc[none, "is_laundering"] == 1, "UNSTRUCTURED", "NONE")
        src = DATA_PATH.name
    rates = implied_usd_rates(df)
    df["amount_usd"] = df["amount_paid"] / df["payment_currency"].astype(str).map(rates)
    return df.drop(columns=["amount_received"]), src, n_raw, rates


def load_aws():
    cols = ["ts", "from_bank", "from_account", "to_bank", "to_account", "amount_paid", "amount_paid_usd",
            "receiving_currency", "payment_currency", "payment_format", "is_laundering", "pattern", "attempt_id"]
    df = pd.read_parquet(CURATED_DIR, columns=cols).rename(columns={"amount_paid_usd": "amount_usd"})
    df["ts"] = pd.to_datetime(df["ts"]).astype("datetime64[ns]")
    rates = (df["amount_paid"] / df["amount_usd"]).groupby(df["payment_currency"].astype(str)).median()
    return df, f"{CURATED_DIR} (Parquet)", None, rates


def compact(df):
    """Shared integer ids for accounts, shared categories for banks, float32 / category dtypes."""
    fa, ta = df["from_account"].astype("category"), df["to_account"].astype("category")
    accts = fa.cat.categories.union(ta.cat.categories)
    out = pd.DataFrame({
        "ts": df["ts"].values,
        "from_acct": accts.get_indexer(fa.cat.categories)[fa.cat.codes].astype("int32"),
        "to_acct": accts.get_indexer(ta.cat.categories)[ta.cat.codes].astype("int32"),
    })
    banks = pd.Index(df["from_bank"].astype(str).unique()).union(pd.Index(df["to_bank"].astype(str).unique()))
    for c in ("from_bank", "to_bank"):
        out[c] = pd.Categorical(df[c].astype(str), categories=banks)
    out["amount_paid"] = df["amount_paid"].values.astype("float64")           # native currency
    out["amount_usd"] = df["amount_usd"].values.astype("float32")
    for c in ("payment_format", "payment_currency", "receiving_currency", "pattern"):
        out[c] = df[c].astype(str).astype("category").values
    out["is_laundering"] = df["is_laundering"].values.astype("int8")
    out["attempt_id"] = df["attempt_id"].values.astype("float32")
    return out, len(accts)


t = time.time()
raw, SOURCE, n_raw, FX_RATES = load_local() if AML_MODE == "local" else load_aws()
df, n_accounts = compact(raw)
del raw

print(f"source: {SOURCE}  ({time.time() - t:.0f} s)")
if n_raw is not None:
    print(f"raw rows {n_raw:,}, exact duplicates dropped {n_raw - len(df):,}")
print(f"rows {len(df):,}, laundering {int(df.is_laundering.sum()):,}, accounts {n_accounts:,}")
assert df["amount_usd"].notna().all(), "currency without a USD rate"
assert len(df) == EXPECTED["rows_after_dedup"], f"rows after de-duplication: {len(df):,}"
assert int(df.is_laundering.sum()) == EXPECTED["laundering"], "laundering rows"
assert n_accounts == EXPECTED["accounts"], f"accounts: {n_accounts:,}"
print("\nUnits per USD (simulator's own rates):")
print(FX_RATES.round(6).to_string())

In [ ]:
# Drop the 11-18 Sep tail, then number the transfers in time order
in_tail = df["ts"] >= TAIL_START
print(f"tail rows dropped: {int(in_tail.sum()):,} ({100 * df.loc[in_tail, 'is_laundering'].mean():.1f}% laundering)")
df = df[~in_tail].sort_values("ts", kind="stable").reset_index(drop=True)
df["txn_id"] = np.arange(len(df), dtype="int32")          # index == txn_id from here on

print(df["pattern"].value_counts().to_string())

## 3. Split by time

| Window | Days | Use |
|---|---|---|
| `fit` | 1–6 Sep | Training. |
| `valid` | 7 Sep | Early stopping only. |
| `train` = fit + valid | 1–7 Sep | Fits the account-level features and the alert threshold. |
| `test` | 8–10 Sep | Final evaluation only. |

In [ ]:
df["window"] = np.where(df["ts"] < TEST_START, "train", "test")
is_train = (df["window"] == "train").values
is_test = ~is_train
is_fit = (df["ts"] < VALID_START).values
is_valid = is_train & ~is_fit

split = pd.DataFrame({
    "rows": [is_fit.sum(), is_valid.sum(), is_train.sum(), is_test.sum()],
    "laundering": [int(df.is_laundering.values[m].sum()) for m in (is_fit, is_valid, is_train, is_test)],
}, index=["fit 1-6", "valid 7", "train 1-7", "test 8-10"])
print(split.to_string())
assert split.at["train 1-7", "rows"] == EXPECTED["train_rows"]
assert split.at["train 1-7", "laundering"] == EXPECTED["train_laundering"]
assert split.at["test 8-10", "laundering"] == EXPECTED["test_laundering"]

## 4. Base features (Experiment A)

**Per-transaction features**

- **Time:** `hour_of_day`, `day_of_week`, `is_weekend`, `is_night` (22:00–05:59).
- **Amount:** `amount` and `log_amount` are in USD. `just_under_10k` uses USD, because the reporting threshold is in USD. `is_round_amount` uses the currency actually paid, because people round in the currency they send.
- **Flags:** `is_cross_bank`, `is_cross_currency`, `is_self_transfer`.
- **Categoricals:** `payment_format`, `payment_currency`.

**Same-day velocity:** `sender_txns_same_day` and `receiver_txns_same_day`. They count within one calendar day, so they never cross the 7 → 8 Sep split.

**One-hop account features:** fan-out and fan-in degree, transaction counts, unique counterparty banks, and pass-through (USD out ÷ (USD in + 1)). Accounts with no history get `NaN`.

**Out-of-day fitting (a change from the handover spec).** The spec fits account features on all of 1–7 Sep and maps them onto every row. A train row's features then already count that row's own transfer, and a laundering account's laundering, while a test row's features cannot. The model learns a shortcut that is gone at test time. On the full run it scored PR-AUC 0.42 on 7 Sep but 0.105 on test.

- **Train rows:** each train day *d* gets features fitted on the other six train days.
- **Test rows:** features are fitted on all of 1–7 Sep.

No row's features ever include its own day, so train and test rows mean the same thing. Test days are never used to fit anything. The two-hop features in section 5 are built the same way.

In [ ]:
t = time.time()
hour = df["ts"].dt.hour
df["hour_of_day"] = hour.astype("int8")
df["day_of_week"] = df["ts"].dt.dayofweek.astype("int8")
df["is_weekend"] = (df["day_of_week"] >= 5).astype("int8")
df["is_night"] = ((hour >= 22) | (hour < 6)).astype("int8")

df["amount"] = df["amount_usd"]
df["log_amount"] = np.log1p(df["amount_usd"]).astype("float32")
df["is_round_amount"] = (np.isclose(df["amount_paid"] % 1000, 0) & (df["amount_paid"] >= 1000)).astype("int8")
df["just_under_10k"] = ((df["amount_usd"] >= 9_000) & (df["amount_usd"] < 10_000)).astype("int8")

df["is_cross_bank"] = (df["from_bank"].cat.codes != df["to_bank"].cat.codes).astype("int8")
df["is_cross_currency"] = (df["payment_currency"].astype(str) != df["receiving_currency"].astype(str)).astype("int8")
df["is_self_transfer"] = (df["from_acct"] == df["to_acct"]).astype("int8")

day = df["ts"].dt.normalize()
df["sender_txns_same_day"] = df.groupby([df["from_acct"], day])["txn_id"].transform("size").astype("int32")
df["receiver_txns_same_day"] = df.groupby([df["to_acct"], day])["txn_id"].transform("size").astype("int32")

SENDER_ACCT = ["sender_fan_out_degree", "sender_out_txn_count", "sender_unique_banks",
               "sender_pass_through_ratio", "sender_two_hop_out"]
RECEIVER_ACCT = ["receiver_fan_in_degree", "receiver_in_txn_count", "receiver_unique_banks",
                 "receiver_two_hop_in"]


def fit_account_features(mask):
    """Per-account one-hop and two-hop features from the rows in mask."""
    tr = df.loc[mask, ["from_acct", "to_acct", "from_bank", "to_bank", "amount_usd", "txn_id"]]
    snd = tr.groupby("from_acct").agg(sender_out_txn_count=("txn_id", "size"), out_usd=("amount_usd", "sum"))
    snd["sender_fan_out_degree"] = tr.drop_duplicates(["from_acct", "to_acct"]).groupby("from_acct").size()
    snd["sender_unique_banks"] = tr.drop_duplicates(["from_acct", "to_bank"]).groupby("from_acct").size()
    rcv = tr.groupby("to_acct").agg(receiver_in_txn_count=("txn_id", "size"), in_usd=("amount_usd", "sum"))
    rcv["receiver_fan_in_degree"] = tr.drop_duplicates(["to_acct", "from_acct"]).groupby("to_acct").size()
    rcv["receiver_unique_banks"] = tr.drop_duplicates(["to_acct", "from_bank"]).groupby("to_acct").size()
    snd["sender_pass_through_ratio"] = snd["out_usd"].astype("float64") / (
        rcv["in_usd"].astype("float64").reindex(snd.index).fillna(0) + 1)
    # two-hop reach (Experiment B): distinct edges, self-loops excluded
    edges = tr.loc[tr["from_acct"] != tr["to_acct"], ["from_acct", "to_acct"]].drop_duplicates()
    fan_out, fan_in = edges.groupby("from_acct").size(), edges.groupby("to_acct").size()
    snd["sender_two_hop_out"] = edges["to_acct"].map(fan_out).fillna(0).groupby(edges["from_acct"]).sum()
    rcv["receiver_two_hop_in"] = edges["from_acct"].map(fan_in).fillna(0).groupby(edges["to_acct"]).sum()
    return snd, rcv


def map_account_features(rows, snd, rcv):
    for c in SENDER_ACCT:
        df.loc[rows, c] = df.loc[rows, "from_acct"].map(snd[c]).astype("float32").values
    for c in RECEIVER_ACCT:
        df.loc[rows, c] = df.loc[rows, "to_acct"].map(rcv[c]).astype("float32").values


for c in SENDER_ACCT + RECEIVER_ACCT:
    df[c] = np.float32(np.nan)
txn_day = df["ts"].dt.day.values
for d in range(1, 8):                                   # train day d <- the other six train days
    map_account_features(is_train & (txn_day == d), *fit_account_features(is_train & (txn_day != d)))
map_account_features(is_test, *fit_account_features(is_train))   # test <- all of 1-7 Sep

FEATURES_A = [
    "hour_of_day", "day_of_week", "is_weekend", "is_night",
    "amount", "log_amount", "is_round_amount", "just_under_10k",
    "is_cross_bank", "is_cross_currency", "is_self_transfer",
    "payment_format", "payment_currency",
    "sender_txns_same_day", "receiver_txns_same_day",
    "sender_fan_out_degree", "sender_out_txn_count", "sender_unique_banks", "sender_pass_through_ratio",
    "receiver_fan_in_degree", "receiver_in_txn_count", "receiver_unique_banks",
]
print(f"{len(FEATURES_A)} Experiment A features ({time.time() - t:.0f} s)")
print("sender with no history: " + ", ".join(
    f"{n} {df.loc[m, 'sender_fan_out_degree'].isna().mean():.1%}" for n, m in (("train", is_train), ("test", is_test))))

## 5. Network features (Experiment B = A + these)

| Feature | Definition | Scope |
|---|---|---|
| `fwd_within_24h` | The receiver sends any transfer in (ts, ts + 24 h]. | Per window. |
| `mins_to_forward` | Minutes until the receiver's next outgoing transfer; `NaN` if none in the window. | Per window. |
| `returns_to_sender_72h` | The receiver sends money back to the sender (B → A) in (ts, ts + 72 h]. | Per window. |
| `sender_two_hop_out` | Sum of fan-out degree over the sender's distinct out-neighbours. | Out-of-day, as in section 4. |
| `receiver_two_hop_in` | Sum of fan-in degree over the receiver's distinct in-neighbours. | Out-of-day, as in section 4. |

**Self-transfers (A → A) are left out** of every lookup and of the two-hop edges. Their rows get `NaN`; otherwise `returns_to_sender_72h` would match a row against itself.

**"After ts" is strict.** A transfer in the same minute does not count, following the (ts, …] definitions; the handover's hint code also counted same-minute matches.

**Each lookup stays inside its own window,** so a 7 Sep transfer is never matched to an 8 Sep one.

**Cut-off-aware (a change from the handover spec).** Near the end of a window, the 24 h or 72 h horizon runs past the window edge. A missing match there means "unknown", not "no". Such rows get `NaN` instead of 0, so a real 0 only appears where the whole horizon was visible. Without this, 7 Sep and 10 Sep look systematically "quieter" than the days the model learns from. In the first full run, B early-stopped at round 179 with a lower 7 Sep score than A.

**The stretch feature `in_3cycle_72h` is not built.** It is an open decision for Qiaoye.

In [ ]:
t = time.time()


def network_features(w, window_end):
    """Forward-looking relay features for one window. w: rows of one window, sorted by ts.
    A flag is 1 if the event happens within its horizon, 0 if it provably does not (a later
    event, or the whole horizon lies inside the window), and NaN if the horizon is cut off."""
    nw = w.loc[w["from_acct"] != w["to_acct"], ["txn_id", "from_acct", "to_acct", "ts"]]

    # next outgoing transfer of the receiver, strictly after ts
    out = nw[["from_acct", "ts"]].rename(columns={"from_acct": "acct", "ts": "next_out_ts"})
    left = nw[["txn_id", "to_acct", "ts"]].rename(columns={"to_acct": "acct"})
    m = pd.merge_asof(left, out, left_on="ts", right_on="next_out_ts", by="acct",
                      direction="forward", allow_exact_matches=False)
    mins = (m["next_out_ts"] - m["ts"]).dt.total_seconds() / 60

    # the reversed pair B -> A, strictly after ts
    back = nw[["to_acct", "from_acct", "ts"]].rename(
        columns={"to_acct": "from_acct", "from_acct": "to_acct", "ts": "back_ts"})
    r = pd.merge_asof(nw, back, left_on="ts", right_on="back_ts", by=["from_acct", "to_acct"],
                      direction="forward", allow_exact_matches=False)

    # merge_asof keeps the left row order, so m, r and nw line up row for row
    assert (m["txn_id"].values == nw["txn_id"].values).all() and (r["txn_id"].values == nw["txn_id"].values).all()

    ts = nw["ts"].values                              # positional: m and r have their own index

    def flag(gap, horizon):
        gap = gap.values
        out = np.array(gap <= horizon, dtype="float32")
        out[np.isnat(gap) & (ts + horizon > window_end)] = np.nan   # unseen, horizon past the edge
        return out

    return pd.DataFrame({
        "mins_to_forward": mins.values,
        "fwd_within_24h": flag(m["next_out_ts"] - m["ts"], pd.Timedelta("24h")),
        "returns_to_sender_72h": flag(r["back_ts"] - r["ts"], pd.Timedelta("72h")),
    }, index=nw["txn_id"].values)


NET_COLS = ["mins_to_forward", "fwd_within_24h", "returns_to_sender_72h"]
for c in NET_COLS:
    df[c] = np.float32(np.nan)                       # self-transfer rows stay NaN
for name, window_end in (("train", TEST_START), ("test", TAIL_START)):
    w = df.loc[df["window"] == name, ["txn_id", "from_acct", "to_acct", "ts"]]
    nf = network_features(w, window_end)
    df.loc[nf.index, NET_COLS] = nf[NET_COLS].astype("float32").values
cut_off = df.loc[df["is_self_transfer"] == 0, ["fwd_within_24h", "returns_to_sender_72h"]].isna().mean()
print("share of non-self rows cut off at a window edge (NaN): " + ", ".join(f"{k} {v:.1%}" for k, v in cut_off.items()))
# sender_two_hop_out / receiver_two_hop_in were built out-of-day in section 4

NETWORK_FEATURES = ["fwd_within_24h", "mins_to_forward", "returns_to_sender_72h",
                    "sender_two_hop_out", "receiver_two_hop_in"]
FEATURES_B = FEATURES_A + NETWORK_FEATURES
print(f"{len(FEATURES_B)} Experiment B features ({time.time() - t:.0f} s)")

# Never a model input: labels, scheme ids, dates, row ids, raw account / bank ids
FORBIDDEN = {"is_laundering", "pattern", "attempt_id", "txn_date", "in_tail",
             "ts", "txn_id", "window", "from_acct", "to_acct", "from_bank", "to_bank"}
assert not FORBIDDEN & set(FEATURES_A), FORBIDDEN & set(FEATURES_A)
assert not FORBIDDEN & set(FEATURES_B), FORBIDDEN & set(FEATURES_B)
assert set(FEATURES_A) < set(FEATURES_B) and len(set(FEATURES_B)) == len(FEATURES_B)
print("feature-list assert passed: no label, pattern, date or id column is an input")

**Signal check.** This is the share of train-window rows (self-transfers excluded) where each relay flag is 1, among rows whose horizon is fully visible. The handover measured these on 12 Oct with the hint code, which counted same-minute matches and treated cut-off rows as 0. Expect small differences.

In [ ]:
nst = df.loc[is_train & (df["is_self_transfer"] == 0).values]
grp = np.where(nst["is_laundering"] == 0, "Normal", nst["pattern"].astype(str))
sig = nst.groupby(grp)[["fwd_within_24h", "returns_to_sender_72h"]].mean().mul(100)
sig.loc["Laundering, all"] = nst.loc[nst["is_laundering"] == 1, ["fwd_within_24h", "returns_to_sender_72h"]].mean() * 100
sig["rows"] = pd.Series(grp).value_counts()
sig.loc["Laundering, all", "rows"] = int((nst["is_laundering"] == 1).sum())
print(sig.round(2).sort_values("returns_to_sender_72h", ascending=False).to_string())
del nst

## 6. Train

Both experiments use the same XGBoost settings, the same split and the same seed. The only difference is the feature list.

**One setting changed from the spec:** `learning_rate=0.1` instead of 0.05. At 0.05, A ran to the 2,000-round cap and took 33 minutes, against the 20-minute budget for the whole notebook.

- **Training:** the model trains on 1–6 Sep and early-stops on PR-AUC for 7 Sep. `scale_pos_weight` is negatives ÷ positives on the fit rows.
- **Scoring:** test rows are scored with the best iteration.
- **No tuning grid:** none was run.

In [ ]:
XGB_PARAMS = dict(n_estimators=2000, learning_rate=0.1, max_depth=6, min_child_weight=5,
                  subsample=0.8, colsample_bytree=0.8, eval_metric="aucpr", tree_method="hist",
                  early_stopping_rounds=100, random_state=SEED, enable_categorical=True, n_jobs=-1)
y = df["is_laundering"].values


def train_eval(feature_list, name):
    t = time.time()
    X_fit, y_fit = df.loc[is_fit, feature_list], y[is_fit]
    X_val, y_val = df.loc[is_valid, feature_list], y[is_valid]
    spw = float((y_fit == 0).sum() / (y_fit == 1).sum())
    model = xgb.XGBClassifier(scale_pos_weight=spw, **XGB_PARAMS)
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=250)
    scores = model.predict_proba(df.loc[is_test, feature_list])[:, 1]
    print(f"[{name}] scale_pos_weight={spw:.0f}, best_iteration={model.best_iteration}, "
          f"valid aucpr={model.best_score:.4f}, {time.time() - t:.0f} s")
    return model, scores


model_A, scores_A = train_eval(FEATURES_A, "A")

In [ ]:
model_B, scores_B = train_eval(FEATURES_B, "B")

## 7. Evaluate on test (8–10 Sep)

- **PR-AUC.**
- **Best F1:** the best minority-class F1 over all thresholds, with its precision and recall.
- **Recall at a fixed alert budget:** the top 0.5% and the top 1% of test rows. The cut-off is count-based, `ceil(share × rows)`.
- **Detection rate per scheme:** at the top 0.5%.

In [ ]:
test = df.loc[is_test, ["pattern", "is_laundering", "payment_format", "payment_currency", "from_bank"]].copy()
y_test = test["is_laundering"].values


def top_mask(scores, share):
    k = math.ceil(share * len(scores))
    idx = np.argsort(-scores, kind="stable")[:k]
    m = np.zeros(len(scores), dtype=bool)
    m[idx] = True
    return m, k


def evaluate(scores, name, model):
    p, r, _ = precision_recall_curve(y_test, scores)
    f1 = np.where(p + r > 0, 2 * p * r / np.maximum(p + r, 1e-12), 0)[:-1]
    i = int(np.argmax(f1))
    top05, k05 = top_mask(scores, ALERT_SHARE)
    top1, _ = top_mask(scores, 0.01)
    metrics = {"experiment": name, "pr_auc": average_precision_score(y_test, scores),
               "best_f1": f1[i], "precision_at_best_f1": p[i], "recall_at_best_f1": r[i],
               "recall_top_0_5pct": y_test[top05].sum() / y_test.sum(),
               "recall_top_1pct": y_test[top1].sum() / y_test.sum(),
               "n_alerts_0_5pct": k05, "best_iteration": model.best_iteration}
    laund = test["is_laundering"] == 1
    det = (pd.DataFrame({"pattern": test.loc[laund, "pattern"].astype(str), "caught": top05[laund.values]})
           .groupby("pattern")["caught"].agg(laundering_rows="size", caught_top_0_5pct="sum").reset_index())
    det["detection_rate_pct"] = 100 * det["caught_top_0_5pct"] / det["laundering_rows"]
    det.insert(0, "experiment", name)
    return metrics, det


mA, detA = evaluate(scores_A, "A", model_A)
mB, detB = evaluate(scores_B, "B", model_B)
metrics = pd.DataFrame([mA, mB])

baseline = pd.Series({"pr_auc": 0.31, "best_f1": 0.34, "recall_top_0_5pct": 0.627},
                     name="handover baseline (HGB, 10-day account features)")
print(pd.concat([metrics.set_index("experiment").T, baseline], axis=1).to_string(float_format=lambda v: f"{v:.4f}"))

**Why A scores below the handover baseline.** The baseline (PR-AUC 0.31, 62.7% in the top 0.5%) most likely used the precomputed account columns in `hi_small_clean.csv`. Those were fitted on all 10 days, so a test row's features already counted that account's test-period laundering.

A quick diagnostic on 13 Oct reproduced this. It used the same features, a 10% sample of normal rows, 300 rounds at a learning rate of 0.1, and four ways of building the account features:

| Account features | A: PR-AUC | A: top 0.5% | B: PR-AUC | B: top 0.5% |
|---|---|---|---|---|
| All 10 days (leaky, like the baseline) | 0.360 | 64.3% | 0.345 | 64.1% |
| Fitted on 1–7 Sep (handover spec) | 0.143 | 45.3% | 0.095 | 36.0% |
| Out-of-day (this notebook) | 0.192 | 44.7% | 0.181 | 44.6% |
| None | 0.151 | 38.1% | 0.170 | 41.3% |

The leaky version lands on the baseline's numbers, so 0.31 is not reachable without seeing the test days. The honest comparison is between A and B in this notebook. The diagnostic's B ran before the cut-off fix in section 5.

In [ ]:
BASELINE_DET = {"UNSTRUCTURED": 35.6, "BIPARTITE": 58.8, "STACK": 67.7, "CYCLE": 69.6, "RANDOM": 80.4,
                "GATHER-SCATTER": 82.9, "FAN-IN": 87.9, "FAN-OUT": 88.3, "SCATTER-GATHER": 94.1}
detection = pd.concat([detA, detB], ignore_index=True)
cmp = detection.pivot(index="pattern", columns="experiment", values="detection_rate_pct")
cmp.insert(0, "laundering_rows", detA.set_index("pattern")["laundering_rows"])
cmp["B_minus_A_pts"] = cmp["B"] - cmp["A"]
cmp["baseline_HGB"] = pd.Series(BASELINE_DET)
print("Detection rate (%) at the top 0.5% of test scores")
print(cmp.sort_values("B_minus_A_pts", ascending=False).round(1).to_string())

**Caveat: forward-looking features.** `fwd_within_24h`, `mins_to_forward` and `returns_to_sender_72h` look forward in time, so a score that uses them is available only 24–72 hours after the transfer. That is acceptable for BQ2, which explains past behaviour. For real-time alerting, Experiment B would be a delayed second-pass score; Experiment A is the real-time one.

## 8. Explain

- **Gain importance:** XGBoost's average gain per split for each feature.
- **SHAP:** mean |SHAP| from `TreeExplainer`, computed on the top 1,000 test alerts plus 5,000 random test rows. If `shap` cannot read the categorical splits, XGBoost's built-in TreeSHAP (`pred_contribs`) is used instead; it is the same algorithm.

In [ ]:
def gain_importance(model, features, name):
    g = model.get_booster().get_score(importance_type="gain")
    out = pd.DataFrame({"experiment": name, "feature": features, "gain": [g.get(f, 0.0) for f in features]})
    out = out.sort_values("gain", ascending=False, ignore_index=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out


rng = np.random.default_rng(SEED)
test_pos = np.arange(is_test.sum())


def shap_summary(model, features, scores, name):
    top = np.argsort(-scores, kind="stable")[:1000]
    rand = rng.choice(np.setdiff1d(test_pos, top), size=5000, replace=False)
    X = df.loc[is_test, features].iloc[np.concatenate([top, rand])]
    try:
        sv = shap.TreeExplainer(model).shap_values(X)
        how = "shap.TreeExplainer"
    except Exception as e:  # categorical splits not supported by this shap version
        dm = xgb.DMatrix(X, enable_categorical=True)
        sv = model.get_booster().predict(dm, pred_contribs=True, iteration_range=(0, model.best_iteration + 1))[:, :-1]
        how = f"xgboost pred_contribs (shap failed: {type(e).__name__})"
    out = pd.DataFrame({"experiment": name, "feature": features, "mean_abs_shap": np.abs(sv).mean(axis=0)})
    out = out.sort_values("mean_abs_shap", ascending=False, ignore_index=True)
    out["rank"] = np.arange(1, len(out) + 1)
    print(f"[{name}] SHAP via {how} on {len(X):,} rows")
    return out


importance = pd.concat([gain_importance(model_A, FEATURES_A, "A"), gain_importance(model_B, FEATURES_B, "B")])
shap_tbl = pd.concat([shap_summary(model_A, FEATURES_A, scores_A, "A"),
                      shap_summary(model_B, FEATURES_B, scores_B, "B")])
side = (shap_tbl[shap_tbl.experiment == "B"].head(15)[["rank", "feature", "mean_abs_shap"]]
        .merge(importance[importance.experiment == "B"][["feature", "rank"]].rename(columns={"rank": "gain_rank"}), on="feature"))
print("Experiment B: top 15 features by mean |SHAP| (gain rank beside)")
print(side.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## 9. Exposure: where the model's risk concentrates

For each group of test transfers, using the Experiment B scores:

- `share_of_score_pct` is the group's share of the summed score.
- `share_of_txns_pct` is its share of transfers.
- `risk_lift` is share of score ÷ share of transfers. Above 1 means the group carries more risk than its volume suggests.
- `actual_laundering` is the group's real laundering count, shown beside the lift for comparison.

In [ ]:
test["score"] = scores_B


def exposure(key):
    g = test.groupby(key, observed=True).agg(txns=("score", "size"), sum_score=("score", "sum"),
                                             actual_laundering=("is_laundering", "sum"))
    g["share_of_txns_pct"] = 100 * g["txns"] / g["txns"].sum()
    g["share_of_score_pct"] = 100 * g["sum_score"] / g["sum_score"].sum()
    g["risk_lift"] = g["share_of_score_pct"] / g["share_of_txns_pct"]
    g = g.reset_index().rename(columns={key: "key"})
    cols = ["key", "txns", "share_of_txns_pct", "sum_score", "share_of_score_pct", "risk_lift", "actual_laundering"]
    return g[cols].sort_values("sum_score", ascending=False, ignore_index=True)


exp_channel = exposure("payment_format")
exp_currency = exposure("payment_currency")
exp_bank = exposure("from_bank").head(100)
print(exp_channel.round(3).to_string(index=False))
print()
print(exp_currency.round(3).head(8).to_string(index=False))
print()
print(exp_bank.round(3).head(10).to_string(index=False))

## 10. Monitoring hand-off

- **`model.json`:** the Experiment B model.
- **`score_threshold.json`:** the score at the top 0.5% cut-off on the train days (1–7 Sep). `monitoring/checks.py::check_anomaly` uses it as `score_threshold`.

The train days include the rows the model was fitted on, so this threshold sits slightly higher than an out-of-sample one would.

In [ ]:
train_scores = model_B.predict_proba(df.loc[is_train, FEATURES_B])[:, 1]
k = math.ceil(ALERT_SHARE * len(train_scores))
threshold = float(np.sort(train_scores)[::-1][k - 1])
model_B.save_model(OUT_DIR / "model.json")
with open(OUT_DIR / "score_threshold.json", "w") as fh:
    json.dump({"alert_share": ALERT_SHARE, "threshold": threshold, "computed_on": "train 1-7 Sep"}, fh, indent=2)
print(f"threshold {threshold:.6f} (top {k:,} of {len(train_scores):,} train rows); "
      f"test rows at or above it: {(scores_B >= threshold).sum():,}")

## 11. Write outputs

In [ ]:
metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
detection.to_csv(OUT_DIR / "detection_by_scheme.csv", index=False)
importance.to_csv(OUT_DIR / "feature_importance.csv", index=False)
shap_tbl.to_csv(OUT_DIR / "shap_summary.csv", index=False)
exp_channel.to_csv(OUT_DIR / "exposure_by_channel.csv", index=False)
exp_currency.to_csv(OUT_DIR / "exposure_by_currency.csv", index=False)
exp_bank.to_csv(OUT_DIR / "exposure_by_bank.csv", index=False)

expected = ["metrics.csv", "detection_by_scheme.csv", "feature_importance.csv", "shap_summary.csv",
            "exposure_by_channel.csv", "exposure_by_currency.csv", "exposure_by_bank.csv",
            "model.json", "score_threshold.json"]
missing = [f for f in expected if not (OUT_DIR / f).exists()]
assert not missing, missing
assert set(pd.read_csv(OUT_DIR / "metrics.csv")["experiment"]) == {"A", "B"}
assert set(pd.read_csv(OUT_DIR / "detection_by_scheme.csv")["experiment"]) == {"A", "B"}
print(f"all {len(expected)} files written to {OUT_DIR}; total run time {(time.time() - T0) / 60:.1f} min")

## Results: what the network features add

Measured on 13 Oct (run time 10.4 min, local mode). Both experiments use the same split, seed and settings.

**Answer.** Following money past one hop helps most for the schemes that move it through chains and loops. At the same 6,728-alert budget (top 0.5% of test rows), detection rises by 8.8 points for bipartite, 8.3 for gather-scatter, 8.1 for stack, 7.6 for cycle and 5.4 for random. Schemes already visible from a single account gain little or lose a little: fan-out +3.2, scatter-gather +1.8 and fan-in −3.3 points. Overall, B lifts PR-AUC from 0.170 to 0.236 and best F1 from 0.256 to 0.301. Unstructured laundering (582 of 1,495 test laundering rows) is still the hardest to catch: 24.1% with A and 22.3% with B.

| Metric (test, 8–10 Sep) | A | B |
|---|---|---|
| PR-AUC | 0.170 | 0.236 |
| Best F1 (precision / recall) | 0.256 (0.260 / 0.252) | 0.301 (0.418 / 0.235) |
| Recall, top 0.5% | 45.4% | 47.9% |
| Recall, top 1% | 55.7% | 58.5% |

**How to read these numbers**

- **Small samples:** each scheme has only 56–181 test laundering rows, so one row is worth 0.6–1.8 points. Differences under about 3 points (fan-out, scatter-gather, fan-in, unstructured) are within noise. The results come from a single seed.
- **Baseline gap:** A sits below the handover baseline (PR-AUC 0.31) because that baseline's account features saw the test days. See the note in section 7.
- **Delayed score:** B's relay features look 24–72 h ahead, so B's score is only available after that delay (see the caveat in section 7).
- **Where risk concentrates:** ACH carries 13.2% of test transfers but 90.6% of B's summed score, and 1,299 of the 1,495 test laundering rows.

In [ ]:
# Upload to S3 when the bucket is ready (needs the aml-bq2 profile):
# aws s3 cp out/model/ s3://<bucket>/marts/model/ --recursive --profile aml-bq2